# **Fashion Designing (less time)**

In [ ]:
# @title Fashion Designing no Face Regenration

!pip install flask pyngrok
!pip install mediapipe opencv-python opencv-python-headless rembg onnxruntime
!pip install diffusers["torch"] transformers accelerate
!pip install git+https://github.com/huggingface/diffusers
!pip install numpy==2.0.0

import io
import os
import cv2
import uuid
import base64
import zipfile
import numpy as np
import torch
import gc
from PIL import Image, ImageOps
from flask import Flask, request, send_file, jsonify, render_template_string
from diffusers import AutoPipelineForInpainting
from rembg import remove
from pyngrok import ngrok

# -------------------------------
# SET UP NGROK & FLASK
# -------------------------------

# Set your ngrok authtoken (replace with your actual token)
ngrok.set_auth_token("2tmGFhW6OOOomrk0n0UIQJzUq5U_4Nur9EZXNMbpD8GArqVi")
app = Flask(__name__)

# -------------------------------
# FRONTEND HTML TEMPLATE
# -------------------------------

HTML_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>AI Fashion Designer</title>
    <!-- Bootstrap CSS -->
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <style>
        body {
            background: linear-gradient(135deg, #f0f2f5, #cfd9df);
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        }
        .container {
            margin-top: 50px;
            max-width: 600px;
            background: white;
            padding: 30px;
            border-radius: 10px;
            box-shadow: 0 10px 25px rgba(0,0,0,0.1);
        }
        .spinner {
            display: none;
            margin: 20px auto;
        }
        .result-img {
            max-width: 100%;
            border-radius: 10px;
            margin-top: 20px;
        }
        .download-btn {
            margin-top: 10px;
        }
        #statusText {
            margin-top: 10px;
            font-weight: bold;
            text-align: center;
        }
    </style>
</head>
<body>
<div class="container">
    <h2 class="text-center mb-4">AI Fashion Designer</h2>
    <form id="designForm" method="post" enctype="multipart/form-data">
        <div class="mb-3">
            <label for="image" class="form-label">Upload Your Image</label>
            <input class="form-control" type="file" id="image" name="image" accept="image/*" required>
        </div>
        <div class="mb-3">
            <label for="prompt" class="form-label">Enter Your Design Prompt</label>
            <input class="form-control" type="text" id="prompt" name="prompt" placeholder="e.g., futuristic, elegant, bold pattern" required>
        </div>
        <button type="submit" class="btn btn-primary w-100">Generate Design</button>
    </form>
    <div class="spinner text-center">
        <div class="spinner-border text-primary" role="status">
          <span class="visually-hidden">Loading...</span>
        </div>
        <div id="statusText">Uploading your image...</div>
    </div>
    <div id="resultSection" class="text-center" style="display:none;">
        <h4 class="mt-4">Your Designed Image</h4>
        <img id="resultImage" src="" alt="Result Image" class="result-img">
        <br>
        <a id="downloadLink" href="#" download="designed_image.png" class="btn btn-success download-btn">Download Image</a>
    </div>
</div>

<!-- JavaScript to handle form submission with progress indications -->
<script>
document.getElementById('designForm').addEventListener('submit', function(e) {
    e.preventDefault();
    var formData = new FormData(this);

    // Hide the form and show the spinner
    document.getElementById('designForm').style.display = 'none';
    document.getElementById('resultSection').style.display = 'none';
    document.querySelector('.spinner').style.display = 'block';
    document.getElementById('statusText').innerText = 'Uploading your image...';

    var xhr = new XMLHttpRequest();
    xhr.open("POST", "/generate", true);

    // Update progress during upload
    xhr.upload.onprogress = function(e) {
        if (e.lengthComputable) {
            var percentComplete = Math.round((e.loaded / e.total) * 100);
            document.getElementById('statusText').innerText = 'Uploading: ' + percentComplete + '%';
        }
    };

    // When the upload is complete, update status to processing
    xhr.upload.onload = function(e) {
        document.getElementById('statusText').innerText = 'Processing your design...';
    };

    // Update progress during download
    xhr.onprogress = function(e) {
        if (e.lengthComputable) {
            var percentComplete = Math.round((e.loaded / e.total) * 100);
            document.getElementById('statusText').innerText = 'Downloading: ' + percentComplete + '%';
        }
    };

    xhr.onload = function() {
        if (xhr.status === 200) {
            document.querySelector('.spinner').style.display = 'none';
            var response = JSON.parse(xhr.responseText);
            if(response.error) {
                alert("Error: " + response.error);
                document.getElementById('designForm').style.display = 'block';
                return;
            }
            // Display the result image
            document.getElementById('resultImage').src = response.image;
            document.getElementById('downloadLink').href = response.image;
            document.getElementById('resultSection').style.display = 'block';
        } else {
            alert("An error occurred during processing.");
            document.querySelector('.spinner').style.display = 'none';
            document.getElementById('designForm').style.display = 'block';
        }
    };

    xhr.onerror = function() {
        alert("An error occurred during processing.");
        document.querySelector('.spinner').style.display = 'none';
        document.getElementById('designForm').style.display = 'block';
    };

    xhr.send(formData);
});
</script>
</body>
</html>
"""

@app.route('/')
def index():
    return render_template_string(HTML_TEMPLATE)

# -------------------------------
# HELPER FUNCTIONS FOR IMAGE PROCESSING
# -------------------------------

def square_image(image: Image.Image) -> Image.Image:
    """Make the image square by adding black borders if needed."""
    width, height = image.size
    if width == height:
        return image
    if width > height:
        top = (width - height) // 2
        bottom = width - height - top
        left = right = 0
    else:
        left = (height - width) // 2
        right = height - width - left
        top = bottom = 0
    return ImageOps.expand(image, border=(left, top, right, bottom), fill="black")

def detect_face(image: Image.Image):
    """Detect face using OpenCV's Haar cascade and return (x1, y1, x2, y2) bbox."""
    cv_image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    faces = face_cascade.detectMultiScale(cv_image, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    if len(faces) > 0:
        x, y, w, h = faces[0]
        return (x, y, x + w, y + h)
    return None

def generate_mask(image: Image.Image) -> Image.Image:
    """
    Remove the background from the image using rembg,
    then detect the face and adjust the mask so that the face (and some extra area) is white,
    and all other areas are black.
    """
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    input_data = buf.getvalue()
    output_data = remove(input_data)
    image_rgba = Image.open(io.BytesIO(output_data)).convert("RGBA")

    # Composite onto a white background
    white_bg = Image.new("RGBA", image_rgba.size, (255, 255, 255, 255))
    image_with_white_bg = Image.alpha_composite(white_bg, image_rgba).convert("RGB")

    # Optionally adjust the mask around a detected face
    face_bbox = detect_face(image_with_white_bg)
    pixel_data = np.array(image_with_white_bg)
    if face_bbox:
        x1, y1, x2, y2 = face_bbox
        # Increase the margin factor to make the face box bigger
        margin_factor = 0.4  # Adjust this factor as needed

        # Calculate margins based on the face's width and height
        height_margin = int((y2 - y1) * margin_factor)
        width_margin = int((x2 - x1) * margin_factor)

        # Expand the bounding box in all directions
        y1 = max(0, y1 - height_margin)
        y2 = min(pixel_data.shape[0], y2 + height_margin)
        x1 = max(0, x1 - width_margin)
        x2 = min(pixel_data.shape[1], x2 + width_margin)

        # Set the expanded face region to white in the mask
        pixel_data[y1:y2, x1:x2] = [255, 255, 255]

    # Create a binary mask: white for background and face region, black elsewhere
    mask = np.all(pixel_data == [255, 255, 255], axis=-1)
    pixel_data[~mask] = [0, 0, 0]
    dilated_mask = cv2.dilate(mask.astype(np.uint8), None, iterations=2).astype(bool)
    pixel_data[dilated_mask] = [255, 255, 255]

    return Image.fromarray(pixel_data)

def invert_colors(image: Image.Image) -> Image.Image:
    """Invert the colors of the image."""
    if image.mode in ('RGB', 'RGBA'):
        channels = image.split()
        if image.mode == 'RGBA':
            r, g, b, a = channels
            rgb_inverted = ImageOps.invert(Image.merge('RGB', (r, g, b)))
            r2, g2, b2 = rgb_inverted.split()
            return Image.merge('RGBA', (r2, g2, b2, a))
        else:
            return ImageOps.invert(image)
    return ImageOps.invert(image)

def inpaint_image(init_image: Image.Image, mask_image: Image.Image, prompt: str, seed=92) -> Image.Image:
    """Use the inpainting pipeline to generate the new image with adjusted parameters for better quality."""
    if torch.cuda.is_available():
        generator = torch.Generator("cuda").manual_seed(seed)
    else:
        generator = torch.Generator().manual_seed(seed)

    result = inpaint_pipe(
        prompt=f"{prompt} dont change the face proper hands",
        image=init_image,
        mask_image=mask_image,
        generator=generator,
        num_inference_steps=100,  # More steps for refined details
        strength=0.8,             # Lower strength preserves more of the original
        guidance_scale=7.5,       # Adjust as needed for prompt adherence
        negative_prompt=("artificial, robotic, android-like, synthetic skin, glossy skin, plastic texture, "
                         "3D render, uncanny valley, croped, lowres, bad anatomy, bad hands, text, error, "
                         "missing fingers, extra digit, fewer digits, cropped, duplicates, worst quality, "
                         "low quality, jpeg artifacts, signature, croped body, watermark, blurry, bad feet, "
                         "mutation, deformed, cross-eyed, malformed limbs, half body, cartoon")
    )
    return result.images[0]

def image_to_base64(image: Image.Image) -> str:
    """Encode a PIL image to a base64 string."""
    buffered = io.BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

# -------------------------------
# LOAD THE DIFFUSERS INPAINTING PIPELINE
# -------------------------------
inpaint_pipe = AutoPipelineForInpainting.from_pretrained(
    "Uminosachi/realisticVisionV51_v51VAE-inpainting", torch_dtype=torch.float16
)
# If you have a GPU available, you can uncomment the next line:
# inpaint_pipe.to("cuda")
inpaint_pipe.enable_model_cpu_offload()
inpaint_pipe.safety_checker = None  # Disable safety checker

# -------------------------------
# ENDPOINT: PROCESS DESIGN REQUEST
# -------------------------------

@app.route('/generate', methods=['POST'])
def generate_endpoint():
    """
    Expects a multipart/form-data POST with:
      - 'image': the image file
      - 'prompt': a text prompt
    This endpoint processes the image and returns the final inpainted image as a base64-encoded PNG.
    """
    if 'image' not in request.files or 'prompt' not in request.form:
        return jsonify({"error": "Please provide both an image file and a prompt."}), 400

    file = request.files['image']
    prompt = request.form['prompt']

    try:
        image = Image.open(file).convert("RGB")
    except Exception:
        return jsonify({"error": "Invalid image file."}), 400

    # Step 1: Make the image square.
    squared_image = square_image(image)

    # Step 2: Generate a mask from the squared image.
    mask_image = generate_mask(squared_image)

    # Step 3: Invert the mask colors.
    mask_image_inverted = invert_colors(mask_image)

    # Preserve face region by setting the face area to black in the inverted mask.
    face_bbox = detect_face(squared_image)
    if face_bbox:
        x1, y1, x2, y2 = face_bbox
        mask_np = np.array(mask_image_inverted)
        mask_np[y1:y2, x1:x2] = 0
        mask_image_inverted = Image.fromarray(mask_np)

    # Step 4: Run the inpainting pipeline.
    result_image = inpaint_image(squared_image, mask_image_inverted, prompt)

    # Convert result image to base64.
    result_base64 = image_to_base64(result_image)
    data_url = f"data:image/png;base64,{result_base64}"

    # Clean up memory by deleting large objects and running garbage collection.
    del image, squared_image, mask_image, mask_image_inverted, result_image
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return jsonify({"image": data_url})

# -------------------------------
# RUN THE APP WITH NGROK
# -------------------------------
if __name__ == '__main__':
    # Open an ngrok tunnel to the HTTP server
    public_url = ngrok.connect(5000)
    print(" * ngrok tunnel \"{}\" -> \"http://127.0.0.1:5000\"".format(public_url))
    # Run the app in Colab
    app.run()


# **Fashion Designing with better Face Quality(takes longer)**

In [ ]:
# @title 1. Install dependencies (12-15 mins)
!pip uninstall torch torchvision -y
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install --upgrade huggingface_hub
!git clone https://github.com/s0md3v/roop.git
%cd roop
!pip install -r requirements.txt
!pip install -q gdown onnxruntime-gpu mxnet-cu90==1.1.0
%cd ..
%cd ..
!pip install --upgrade numpy
!pip install "numpy<2"
!pip install flask pyngrok
!pip install mediapipe opencv-python opencv-python-headless rembg onnxruntime
!pip install triton
!pip install diffusers["torch"] transformers accelerate
!pip install git+https://github.com/huggingface/diffusers

%cd /content/roop
import os
os.makedirs('models', exist_ok=True)
%cd models
# install & download in one go
!pip install -q gdown && gdown https://drive.google.com/uc?id=1krOLgjW2tAPaqV-Bw4YALz0xT5zlb5HF

%cd /content/roop/roop
import os
os.makedirs('models', exist_ok=True)
%cd models
# install & download in one go
!wget -q https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth -O GFPGANv1.4.pth
%cd /content
# @title 1.3. Disable Roop’s NSFW safety check
import io

# Path to the predictor you want to patch
predictor_path = "/content/roop/roop/predictor.py"

# New content that short‑circuits all checks
new_code = '''import threading
# keep imports so nothing breaks downstream
import numpy
import opennsfw2
from PIL import Image
from keras import Model

from roop.typing import Frame

# We disable the predictor entirely by never instantiating it.
PREDICTOR = None
THREAD_LOCK = threading.Lock()

def get_predictor() -> Model:
    # never called
    raise RuntimeError("NSFW predictor is disabled")

def clear_predictor() -> None:
    # no-op
    pass

def predict_frame(target_frame: Frame) -> bool:
    """
    Always returns False (never NSFW).
    """
    return False

def predict_image(target_path: str) -> bool:
    """
    Always returns False (never NSFW).
    """
    return False

def predict_video(target_path: str) -> bool:
    """
    Always returns False (never NSFW).
    """
    return False
'''

# Overwrite the file in one shot
with open(predictor_path, "w", encoding="utf-8") as f:
    f.write(new_code)

print(f"[✓] NSFW predictor disabled in {predictor_path}")



*After block 1 restart the runtime then run the block 1.2*

In [ ]:
# @title 1.2. Install torch (1-2 mins)
!pip uninstall -y torch torchvision torchaudio pytorch-triton triton
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install torchaudio --index-url https://download.pytorch.org/whl/cu121
import torch
import torch.sparse._triton_ops_meta
print("✅ Triton ops are available – torch", torch.__version__)
!pip uninstall basicsr -y
!pip install git+https://github.com/xinntao/BasicSR.git@master
!pip uninstall gfpgan -y
!pip install git+https://github.com/TencentARC/GFPGAN.git@master

In [ ]:
# @title 2. Web UI
import io
import os
import cv2
import uuid
import base64
import zipfile
import numpy as np
import torch
import gc
from PIL import Image, ImageOps
from flask import Flask, request, send_file, jsonify, render_template_string
from diffusers import AutoPipelineForInpainting
from rembg import remove
from pyngrok import ngrok
import shutil
import glob
from werkzeug.utils import secure_filename
import datetime
import time
import subprocess

# -------------------------------
# SET UP NGROK & FLASK
# -------------------------------

# Set your ngrok authtoken (replace with your actual token)
ngrok.set_auth_token("2x0x5HHuUV5Ss7drpzJLbW3s7oa_6prh4t1osTXX22gLMYY2P")
app = Flask(__name__)

# Create necessary directories
UPLOAD_FOLDER = "uploads"
GENERATED_FOLDER = "generated"
SWAPPED_FOLDER = "swapped"
ROOP_DIR = "roop"

for folder in [UPLOAD_FOLDER, GENERATED_FOLDER, SWAPPED_FOLDER]:
    os.makedirs(folder, exist_ok=True)

app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER

# -------------------------------
# FRONTEND HTML TEMPLATE
# -------------------------------

HTML_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>AI Fashion Designer</title>
    <!-- Bootstrap CSS -->
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <style>
        body {
            background: linear-gradient(135deg, #f0f2f5, #cfd9df);
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            min-height: 100vh;
            display: flex;
            align-items: center;
            justify-content: center;
        }
        .container {
            max-width: 800px;
            background: white;
            padding: 40px;
            border-radius: 15px;
            box-shadow: 0 10px 30px rgba(0,0,0,0.1);
            transition: all 0.3s ease;
        }
        .container:hover {
            transform: translateY(-5px);
            box-shadow: 0 15px 35px rgba(0,0,0,0.15);
        }
        .spinner {
            display: none;
            margin: 20px auto;
        }
        .result-img {
            max-width: 100%;
            border-radius: 15px;
            margin-top: 20px;
            box-shadow: 0 5px 15px rgba(0,0,0,0.1);
            transition: all 0.3s ease;
        }
        .result-img:hover {
            transform: scale(1.02);
            box-shadow: 0 8px 20px rgba(0,0,0,0.15);
        }
        .download-btn {
            margin-top: 15px;
            padding: 12px 30px;
            font-weight: 600;
            text-transform: uppercase;
            letter-spacing: 1px;
            transition: all 0.3s ease;
        }
        .download-btn:hover {
            transform: translateY(-2px);
            box-shadow: 0 5px 15px rgba(0,0,0,0.1);
        }
        #statusText {
            margin-top: 15px;
            font-weight: 600;
            text-align: center;
            color: #495057;
        }
        .progress-container {
            margin-top: 25px;
            display: none;
            opacity: 0;
            transition: opacity 0.3s ease;
        }
        .progress {
            height: 10px;
            border-radius: 5px;
            background-color: #e9ecef;
            overflow: hidden;
        }
        .progress-bar {
            background: linear-gradient(45deg, #0d6efd, #0dcaf0);
            transition: width 0.3s ease;
        }
        #progressText {
            margin-top: 10px;
            font-weight: 500;
            color: #6c757d;
        }
        .step-indicator {
            margin-top: 25px;
            display: none;
            opacity: 0;
            transition: opacity 0.3s ease;
        }
        .step {
            padding: 12px 20px;
            margin: 8px 0;
            border-radius: 8px;
            background: #f8f9fa;
            transition: all 0.3s ease;
            display: flex;
            align-items: center;
            gap: 10px;
        }
        .step::before {
            content: '';
            width: 20px;
            height: 20px;
            border-radius: 50%;
            background: #e9ecef;
            display: inline-block;
        }
        .step.active {
            background: #e9ecef;
            border-left: 4px solid #0d6efd;
            transform: translateX(5px);
        }
        .step.active::before {
            background: #0d6efd;
        }
        .step.completed {
            background: #d1e7dd;
            border-left: 4px solid #198754;
        }
        .step.completed::before {
            background: #198754;
        }
        .form-control {
            padding: 12px;
            border-radius: 8px;
            border: 2px solid #e9ecef;
            transition: all 0.3s ease;
        }
        .form-control:focus {
            border-color: #0d6efd;
            box-shadow: 0 0 0 0.2rem rgba(13, 110, 253, 0.15);
        }
        .btn-primary {
            padding: 12px 30px;
            font-weight: 600;
            text-transform: uppercase;
            letter-spacing: 1px;
            transition: all 0.3s ease;
        }
        .btn-primary:hover {
            transform: translateY(-2px);
            box-shadow: 0 5px 15px rgba(13, 110, 253, 0.2);
        }
        .fade-out {
            opacity: 0;
            pointer-events: none;
        }
        .fade-in {
            opacity: 1;
            pointer-events: auto;
        }
    </style>
</head>
<body>
<div class="container">
    <h2 class="text-center mb-4">AI Fashion Designer</h2>
    <form id="designForm" method="post" enctype="multipart/form-data">
        <div class="mb-4">
            <label for="image" class="form-label">Upload Your Image</label>
            <input class="form-control" type="file" id="image" name="image" accept="image/*" required>
        </div>
        <div class="mb-4">
            <label for="prompt" class="form-label">Enter Your Design Prompt</label>
            <input class="form-control" type="text" id="prompt" name="prompt" placeholder="e.g., futuristic, elegant, bold pattern" required>
        </div>
        <button type="submit" class="btn btn-primary w-100">Generate Design</button>
    </form>

    <div class="spinner text-center">
        <div class="spinner-border text-primary" role="status">
          <span class="visually-hidden">Loading...</span>
        </div>
        <div id="statusText">Uploading your image...</div>
    </div>

    <div class="progress-container">
        <div class="progress">
            <div class="progress-bar progress-bar-striped progress-bar-animated" role="progressbar" style="width: 0%"></div>
        </div>
        <div class="text-center mt-2" id="progressText">0%</div>
        <div class="text-center mt-1 text-muted small" id="progressDetails"></div>
    </div>

    <div class="step-indicator">
        <div class="step" id="step1">1. Uploading Image</div>
        <div class="step" id="step2">2. Processing Image</div>
        <div class="step" id="step3">3. Generating Design</div>
        <div class="step" id="step4">4. Face Swap</div>
        <div class="step" id="step5">5. Finalizing</div>
    </div>

    <div id="resultSection" class="text-center" style="display:none;">
        <h4 class="mt-4">Your Designed Image</h4>
        <img id="resultImage" src="" alt="Result Image" class="result-img">
        <br>
        <a id="downloadLink" href="#" download="designed_image.png" class="btn btn-success download-btn">Download Image</a>
    </div>
</div>

<!-- JavaScript to handle form submission with progress indications -->
<script>
let progressInterval;

function updateStep(stepNumber, status) {
    const step = document.getElementById(`step${stepNumber}`);
    step.className = `step ${status}`;
}

function updateProgress(percent, message) {
    const progressBar = document.querySelector('.progress-bar');
    const progressText = document.getElementById('progressText');
    progressBar.style.width = `${percent}%`;
    progressText.textContent = `${percent}% - ${message}`;
}

function showProgressElements() {
    const progressContainer = document.querySelector('.progress-container');
    const stepIndicator = document.querySelector('.step-indicator');

    // Reset classes first
    progressContainer.classList.remove('fade-out');
    stepIndicator.classList.remove('fade-out');

    // Show elements
    progressContainer.style.display = 'block';
    stepIndicator.style.display = 'block';

    // Force reflow
    progressContainer.offsetHeight;

    // Add fade-in class
    progressContainer.classList.add('fade-in');
    stepIndicator.classList.add('fade-in');
}

function hideProgressElements() {
    const progressContainer = document.querySelector('.progress-container');
    const stepIndicator = document.querySelector('.step-indicator');

    // Add fade-out class
    progressContainer.classList.add('fade-out');
    stepIndicator.classList.add('fade-out');

    // Hide after animation
    setTimeout(() => {
        progressContainer.style.display = 'none';
        stepIndicator.style.display = 'none';
        // Reset classes
        progressContainer.classList.remove('fade-in', 'fade-out');
        stepIndicator.classList.remove('fade-in', 'fade-out');
    }, 300);
}

function resetProgressState() {
    // Reset progress bar
    updateProgress(0, 'Starting...');

    // Reset all steps
    for (let i = 1; i <= 5; i++) {
        updateStep(i, '');
    }

    // Reset progress container and step indicator
    const progressContainer = document.querySelector('.progress-container');
    const stepIndicator = document.querySelector('.step-indicator');

    progressContainer.classList.remove('fade-in', 'fade-out');
    stepIndicator.classList.remove('fade-in', 'fade-out');
}

async function resetServerProgress() {
    try {
        await fetch('/reset-progress', { method: 'POST' });
    } catch (error) {
        console.error('Error resetting server progress:', error);
    }
}

function startProgressTracking() {
    // Reset state before starting
    resetProgressState();
    resetServerProgress();

    // Show progress elements
    showProgressElements();

    // Start first step
    updateStep(1, 'active');

    progressInterval = setInterval(() => {
        fetch('/progress')
            .then(response => response.json())
            .then(data => {
                if (data.status === 'error') {
                    clearInterval(progressInterval);
                    hideProgressElements();
                    return;
                }

                // Update progress
                updateProgress(data.progress, data.message);

                // Update details if available
                const progressDetails = document.getElementById('progressDetails');
                if (data.details) {
                    progressDetails.textContent = data.details;
                    progressDetails.style.display = 'block';
                } else {
                    progressDetails.style.display = 'none';
                }

                // Update steps based on current step
                switch(data.step) {
                    case 'uploading':
                        updateStep(1, 'active');
                        break;
                    case 'processing':
                        updateStep(1, 'completed');
                        updateStep(2, 'active');
                        break;
                    case 'generating':
                        updateStep(2, 'completed');
                        updateStep(3, 'active');
                        break;
                    case 'face_swap':
                        updateStep(3, 'completed');
                        updateStep(4, 'active');
                        break;
                    case 'finalizing':
                        updateStep(4, 'completed');
                        updateStep(5, 'active');
                        break;
                    case 'complete':
                        updateStep(5, 'completed');
                        hideProgressElements();
                        break;
                    case 'error':
                        hideProgressElements();
                        break;
                }
            })
            .catch(error => {
                console.error('Error fetching progress:', error);
                hideProgressElements();
            });
    }, 500);
}

function stopProgressTracking() {
    if (progressInterval) {
        clearInterval(progressInterval);
        progressInterval = null;
    }
}

document.getElementById('designForm').addEventListener('submit', function(e) {
    e.preventDefault();
    var formData = new FormData(this);

    // Reset UI
    document.getElementById('designForm').style.display = 'none';
    document.getElementById('resultSection').style.display = 'none';
    document.querySelector('.spinner').style.display = 'block';

    // Start progress tracking
    startProgressTracking();

    var xhr = new XMLHttpRequest();
    xhr.open("POST", "/generate", true);

    // Update progress during upload
    xhr.upload.onprogress = function(e) {
        if (e.lengthComputable) {
            var percentComplete = Math.round((e.loaded / e.total) * 100);
            updateProgress(percentComplete, 'Uploading image...');
        }
    };

    xhr.onload = function() {
        stopProgressTracking();

        if (xhr.status === 200) {
            document.querySelector('.spinner').style.display = 'none';
            var response = JSON.parse(xhr.responseText);

            if(response.error) {
                alert("Error: " + response.error);
                document.getElementById('designForm').style.display = 'block';
                return;
            }

            // Display the result image
            document.getElementById('resultImage').src = response.image;
            document.getElementById('downloadLink').href = response.image;
            document.getElementById('resultSection').style.display = 'block';
            updateProgress(100, 'Complete!');
        } else {
            alert("An error occurred during processing.");
            document.querySelector('.spinner').style.display = 'none';
            document.getElementById('designForm').style.display = 'block';
        }
    };

    xhr.onerror = function() {
        stopProgressTracking();
        alert("An error occurred during processing.");
        document.querySelector('.spinner').style.display = 'none';
        document.getElementById('designForm').style.display = 'block';
    };

    xhr.send(formData);
});
</script>
</body>
</html>
"""

@app.route('/')
def index():
    return render_template_string(HTML_TEMPLATE)

# -------------------------------
# HELPER FUNCTIONS FOR IMAGE PROCESSING
# -------------------------------

def square_image(image: Image.Image) -> Image.Image:
    """Make the image square by adding black borders if needed."""
    width, height = image.size
    if width == height:
        return image
    if width > height:
        top = (width - height) // 2
        bottom = width - height - top
        left = right = 0
    else:
        left = (height - width) // 2
        right = height - width - left
        top = bottom = 0
    return ImageOps.expand(image, border=(left, top, right, bottom), fill="black")

def detect_face(image: Image.Image):
    """Detect face using OpenCV's Haar cascade with caching."""
    # Convert image to bytes for caching
    img_bytes = image.tobytes()

    # Check cache
    if img_bytes in face_detection_cache:
        return face_detection_cache[img_bytes]

    # Detect face
    cv_image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    faces = face_cascade.detectMultiScale(cv_image, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    if len(faces) > 0:
        x, y, w, h = faces[0]
        result = (x, y, x + w, y + h)
        # Cache the result
        face_detection_cache[img_bytes] = result
        return result
    return None

def generate_mask(image: Image.Image) -> Image.Image:
    """
    Remove the background from the image using rembg,
    then detect the face and adjust the mask so that the face (and some extra area) is white,
    and all other areas are black.
    """
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    input_data = buf.getvalue()
    output_data = remove(input_data)
    image_rgba = Image.open(io.BytesIO(output_data)).convert("RGBA")

    # Composite onto a white background
    white_bg = Image.new("RGBA", image_rgba.size, (255, 255, 255, 255))
    image_with_white_bg = Image.alpha_composite(white_bg, image_rgba).convert("RGB")

    # Optionally adjust the mask around a detected face
    face_bbox = detect_face(image_with_white_bg)
    pixel_data = np.array(image_with_white_bg)
    if face_bbox:
        x1, y1, x2, y2 = face_bbox
        # Increase the margin factor to make the face box bigger
        margin_factor = 0.4  # Adjust this factor as needed

        # Calculate margins based on the face's width and height
        height_margin = int((y2 - y1) * margin_factor)
        width_margin = int((x2 - x1) * margin_factor)

        # Expand the bounding box in all directions
        y1 = max(0, y1 - height_margin)
        y2 = min(pixel_data.shape[0], y2 + height_margin)
        x1 = max(0, x1 - width_margin)
        x2 = min(pixel_data.shape[1], x2 + width_margin)

        # Set the expanded face region to white in the mask
        pixel_data[y1:y2, x1:x2] = [255, 255, 255]

    # Create a binary mask: white for background and face region, black elsewhere
    mask = np.all(pixel_data == [255, 255, 255], axis=-1)
    pixel_data[~mask] = [0, 0, 0]
    dilated_mask = cv2.dilate(mask.astype(np.uint8), None, iterations=2).astype(bool)
    pixel_data[dilated_mask] = [255, 255, 255]

    return Image.fromarray(pixel_data)

def invert_colors(image: Image.Image) -> Image.Image:
    """Invert the colors of the image."""
    if image.mode in ('RGB', 'RGBA'):
        channels = image.split()
        if image.mode == 'RGBA':
            r, g, b, a = channels
            rgb_inverted = ImageOps.invert(Image.merge('RGB', (r, g, b)))
            r2, g2, b2 = rgb_inverted.split()
            return Image.merge('RGBA', (r2, g2, b2, a))
        else:
            return ImageOps.invert(image)
    return ImageOps.invert(image)

def inpaint_image(init_image: Image.Image, mask_image: Image.Image, prompt: str, seed=92) -> Image.Image:
    """Use the inpainting pipeline to generate the new image with adjusted parameters for better quality."""
    if torch.cuda.is_available():
        generator = torch.Generator("cuda").manual_seed(seed)
    else:
        generator = torch.Generator().manual_seed(seed)

    result = inpaint_pipe(
        prompt=f"{prompt} dont change the face proper hands",
        image=init_image,
        mask_image=mask_image,
        generator=generator,
        num_inference_steps=100,  # Increased from 80 to 100 steps
        strength=0.8,             # Lower strength preserves more of the original
        guidance_scale=7.5,       # Adjust as needed for prompt adherence
        negative_prompt=("artificial, robotic, android-like, synthetic skin, glossy skin, plastic texture, "
                         "3D render, uncanny valley, cropped, lowres, bad anatomy, bad hands, bad fingers, text, error, "
                         "missing fingers, missing limbs, extra digit, extra fingers, extra hand, extra limbs, multiple arms, multiple legs, "
                         "three hands, duplicated limbs, dislocated joints, broken limbs, fused limbs, malformed hands, malformed limbs, "
                         "wrong limb count, anatomically incorrect, asymmetrical body, twisted pose, unnatural pose, distorted proportions, "
                         "duplicates, worst quality, low quality, jpeg artifacts, signature, cropped body, watermark, blurry, bad feet, "
                         "mutation, deformed, cross-eyed, malformed facial features, lopsided face, lazy eye, misshapen head, "
                         "half body, cartoon, flat colors, childish drawing, sketch, painting style")
    )
    return result.images[0]

def image_to_base64(image: Image.Image) -> str:
    """Encode a PIL image to a base64 string."""
    buffered = io.BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

# -------------------------------
# LOAD THE DIFFUSERS INPAINTING PIPELINE
# -------------------------------
inpaint_pipe = AutoPipelineForInpainting.from_pretrained(
    "Uminosachi/realisticVisionV51_v51VAE-inpainting", torch_dtype=torch.float16
)
"""
inpaint_pipe = AutoPipelineForInpainting.from_pretrained(
    "nesaorg/ClothSwap", torch_dtype=torch.float16
)"""
# If you have a GPU available, you can uncomment the next line:
# inpaint_pipe.to("cuda")
inpaint_pipe.enable_model_cpu_offload()
inpaint_pipe.safety_checker = None  # Disable safety checker

# -------------------------------
# HELPER FUNCTIONS FOR FACE SWAPPING
# -------------------------------

def check_cuda_availability():
    """Check CUDA availability and print diagnostic information."""
    print("\nCUDA Diagnostics:")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA device count: {torch.cuda.device_count()}")
        print(f"Current CUDA device: {torch.cuda.current_device()}")
        print(f"CUDA device name: {torch.cuda.get_device_name(0)}")
        print(f"CUDA version: {torch.version.cuda}")
    return torch.cuda.is_available()

def run_face_swap(source_path, target_path, output_path):
    """Run the face swap process using ROOP."""
    try:
        print(f"Running face swap...")

        # Check CUDA availability
        cuda_available = check_cuda_availability()

        # Set environment variables for CUDA
        os.environ['CUDA_VISIBLE_DEVICES'] = '0'
        os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

        # Set PyTorch to use CUDA if available
        if cuda_available:
            print("Using GPU acceleration through PyTorch...")
            os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'
            os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
            os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'  # For T4 GPU
            os.environ['CUDA_VISIBLE_DEVICES'] = '0'

            # Force CUDA support in ONNX Runtime
            os.environ['ONNXRUNTIME_PROVIDER_CUDA'] = '1'
            os.environ['FORCE_CUDA'] = '1'

        # Use CPU execution provider but enable GPU acceleration through PyTorch environment vars
        cmd = (
            f"CUDA_VISIBLE_DEVICES=0 "
            f"PYTORCH_CUDA_ALLOC_CONF=max_split_size_mb:512 "
            f"CUDA_LAUNCH_BLOCKING=1 "
            f"TORCH_CUDA_ARCH_LIST=7.5 "
            f"ONNXRUNTIME_PROVIDER_CUDA=1 "
            f"FORCE_CUDA=1 "
            f"python {ROOP_DIR}/run.py"
            f" -s \"{source_path}\""
            f" -t \"{target_path}\""
            f" -o \"{output_path}\""
            f" --execution-provider cpu"  # Use CPU as provider but force CUDA through env vars
            f" --frame-processor face_swapper face_enhancer"
            f" --keep-frames"
            f" --skip-audio"
            f" --temp-frame-format jpg"
            f" --output-video-quality 0"
            f" --execution-threads 8"
            f" --max-memory 8192"
        )
        print(f"Command: {cmd}")

        # Run command and capture output
        process = subprocess.Popen(
            cmd,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            universal_newlines=True,
            env=dict(os.environ,
                     CUDA_VISIBLE_DEVICES='0',
                     ONNXRUNTIME_PROVIDER_CUDA='1',
                     FORCE_CUDA='1')  # Ensure CUDA is used
        )
        stdout, stderr = process.communicate()

        if process.returncode != 0:
            print("Face swap failed with error:")
            print(stderr)
            result = process.returncode
        else:
            result = 0

        if result != 0:
            raise Exception(f"Face swap failed with exit code {result}")

        if not os.path.exists(output_path):
            raise Exception("Face swap output file not created")

        # Verify the output file is valid
        try:
            with Image.open(output_path) as img:
                img.verify()
        except Exception as e:
            raise Exception(f"Generated face swap file is invalid: {str(e)}")

        return True
    except Exception as e:
        print(f"Face swap error: {str(e)}")
        # Clean up any partial output
        if os.path.exists(output_path):
            try:
                os.remove(output_path)
            except:
                pass
        return False

# Add face detection cache
face_detection_cache = {}

def detect_face(image: Image.Image):
    """Detect face using OpenCV's Haar cascade with caching."""
    # Convert image to bytes for caching
    img_bytes = image.tobytes()

    # Check cache
    if img_bytes in face_detection_cache:
        return face_detection_cache[img_bytes]

    # Detect face
    cv_image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    faces = face_cascade.detectMultiScale(cv_image, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    if len(faces) > 0:
        x, y, w, h = faces[0]
        result = (x, y, x + w, y + h)
        # Cache the result
        face_detection_cache[img_bytes] = result
        return result
    return None

# -------------------------------
# ENDPOINT: PROCESS DESIGN REQUEST
# -------------------------------

# Add global progress tracking
current_progress = {
    'step': 'idle',
    'progress': 0,
    'message': '',
    'details': ''
}

def reset_progress():
    """Reset the global progress state."""
    global current_progress
    current_progress = {
        'step': 'idle',
        'progress': 0,
        'message': '',
        'details': ''
    }

def update_progress(step, progress, message, details=''):
    """Update the current progress information."""
    global current_progress
    current_progress = {
        'step': step,
        'progress': progress,
        'message': message,
        'details': details
    }

@app.route('/progress', methods=['GET'])
def get_progress():
    """Return the current progress of the image processing."""
    global current_progress
    return jsonify(current_progress)

@app.route('/reset-progress', methods=['POST'])
def reset_progress_endpoint():
    """Reset the progress state."""
    reset_progress()
    return jsonify({'status': 'success'})

@app.route('/generate', methods=['POST'])
def generate_endpoint():
    """
    Expects a multipart/form-data POST with:
      - 'image': the image file
      - 'prompt': a text prompt
    This endpoint processes the image and returns the final inpainted image as a base64-encoded PNG.
    """
    global current_progress

    # Reset progress at the start of each request
    reset_progress()

    if 'image' not in request.files or 'prompt' not in request.form:
        return jsonify({"error": "Please provide both an image file and a prompt."}), 400

    file = request.files['image']
    prompt = request.form['prompt']

    # Initialize paths
    original_path = None
    generated_path = None
    swapped_path = None

    # Create archive folder for saving results
    archive_folder = "archive"
    os.makedirs(archive_folder, exist_ok=True)
    timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    session_folder = os.path.join(archive_folder, timestamp)
    os.makedirs(session_folder, exist_ok=True)

    try:
        # Reset progress
        update_progress('uploading', 0, 'Starting upload...', 'Initializing image processing pipeline')

        # Save the original image
        original_filename = secure_filename(file.filename)
        timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        original_path = os.path.join(UPLOAD_FOLDER, f"{timestamp}_{original_filename}")
        file.save(original_path)

        # Also save a copy to the archive
        archive_original = os.path.join(session_folder, "original.jpg")
        shutil.copy2(original_path, archive_original)

        # Verify the image was saved and can be opened
        if not os.path.exists(original_path):
            raise Exception("Failed to save uploaded image")

        update_progress('processing', 20, 'Loading and processing image...', 'Converting image format and preparing for AI processing')
        image = Image.open(original_path).convert("RGB")

        # Step 1: Make the image square
        update_progress('processing', 30, 'Squaring image...', 'Ensuring uniform dimensions for consistent AI processing')
        squared_image = square_image(image)

        # Step 2: Generate a mask from the squared image
        update_progress('processing', 40, 'Generating mask...', 'Creating background removal mask with rembg')
        mask_image = generate_mask(squared_image)

        # Save mask to archive
        mask_path = os.path.join(session_folder, "mask.png")
        mask_image.save(mask_path)

        # Step 3: Invert the mask colors
        update_progress('processing', 50, 'Processing mask...', 'Inverting mask for inpainting')
        mask_image_inverted = invert_colors(mask_image)

        # Save inverted mask to archive
        inv_mask_path = os.path.join(session_folder, "inverted_mask.png")
        mask_image_inverted.save(inv_mask_path)

        # Preserve face region
        update_progress('processing', 60, 'Detecting and preserving face...', 'Using Haar cascade for facial detection')
        face_bbox = detect_face(squared_image)
        if face_bbox:
            x1, y1, x2, y2 = face_bbox
            mask_np = np.array(mask_image_inverted)
            mask_np[y1:y2, x1:x2] = 0
            mask_image_inverted = Image.fromarray(mask_np)

            # Save face-aware mask
            face_mask_path = os.path.join(session_folder, "face_mask.png")
            mask_image_inverted.save(face_mask_path)

        # Step 4: Run the inpainting pipeline
        update_progress('generating', 70, f'Generating new design with prompt: "{prompt}"...', 'Running AI image generation with Stable Diffusion inpainting')
        result_image = inpaint_image(squared_image, mask_image_inverted, prompt)

        # Save the generated image
        update_progress('generating', 80, 'Saving generated image...', 'Applying final touches to the generated design')
        generated_path = os.path.join(GENERATED_FOLDER, f"generated_{timestamp}.png")
        result_image.save(generated_path)

        # Also save to archive
        archive_generated = os.path.join(session_folder, "generated.png")
        result_image.save(archive_generated)

        # Verify the generated image exists
        if not os.path.exists(generated_path):
            raise Exception("Failed to save generated image")

        # Step 5: Apply face swap using ROOP
        update_progress('face_swap', 85, 'Starting face swap...', 'Using ROOP AI for face swapping with CUDA acceleration')
        swapped_path = os.path.join(SWAPPED_FOLDER, f"swapped_{timestamp}.png")

        # Verify ROOP directory exists
        if not os.path.exists(ROOP_DIR):
            raise Exception(f"ROOP directory not found at {ROOP_DIR}")

        # Run face swap with retries
        max_retries = 3
        for attempt in range(max_retries):
            update_progress('face_swap', 85 + (attempt * 5), f'Face swap attempt {attempt + 1}/{max_retries}...', 'Using GPU-accelerated face detection and alignment')
            if run_face_swap(original_path, generated_path, swapped_path):
                # Save to archive if successful
                archive_swapped = os.path.join(session_folder, "final_swapped.png")
                shutil.copy2(swapped_path, archive_swapped)
                break
            print(f"Face swap attempt {attempt + 1} failed, {'retrying...' if attempt < max_retries - 1 else 'using generated image'}")
            if attempt < max_retries - 1:
                time.sleep(1)  # Wait a bit before retrying
        else:
            print("All face swap attempts failed, using generated image")
            swapped_path = generated_path
            # Save the generated image as final in this case
            archive_final = os.path.join(session_folder, "final_generated.png")
            shutil.copy2(generated_path, archive_final)

        # Convert final result image to base64
        update_progress('finalizing', 95, 'Finalizing image...', 'Preparing image for display and download')
        final_image = Image.open(swapped_path)
        result_base64 = image_to_base64(final_image)
        data_url = f"data:image/png;base64,{result_base64}"

        # Save prompt information
        prompt_file = os.path.join(session_folder, "prompt.txt")
        with open(prompt_file, "w") as f:
            f.write(f"Original image: {original_filename}\n")
            f.write(f"Timestamp: {timestamp}\n")
            f.write(f"Prompt: {prompt}\n")
            f.write(f"Processing completed: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

        # Clean up temporary files but keep the archive
        for path in [original_path, generated_path]:
            if os.path.exists(path):
                os.remove(path)

        # Clean up memory
        del image, squared_image, mask_image, mask_image_inverted, result_image, final_image
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        update_progress('complete', 100, 'Processing completed successfully', f'All images saved to {session_folder}')
        return jsonify({
            "image": data_url,
            "status": "success",
            "message": "Processing completed successfully",
            "archive_path": session_folder
        })

    except Exception as e:
        # Log the error
        print(f"Error in generate_endpoint: {str(e)}")
        import traceback
        traceback.print_exc()

        # Clean up on error
        for path in [original_path, generated_path, swapped_path]:
            if path and os.path.exists(path):
                try:
                    os.remove(path)
                except:
                    pass

        update_progress('error', 0, f'Error: {str(e)}')
        return jsonify({
            "error": f"Error during processing: {str(e)}",
            "status": "error"
        }), 500

# Add favicon route to prevent 404
@app.route('/favicon.ico')
def favicon():
    return '', 204  # Return no content for favicon requests

# -------------------------------
# RUN THE APP WITH NGROK
# -------------------------------
if __name__ == '__main__':
    # Open an ngrok tunnel to the HTTP server
    public_url = ngrok.connect(5000)
    print(" * ngrok tunnel \"{}\" -> \"http://127.0.0.1:5000\"".format(public_url))
    # Run the app in Colab
    app.run()
